# Track 2 · Stage 4 — Evaluation

## Run **GATE 3 first**, before any generation or training.

`whisper-large-v2` zero-shot on the test set should score **≈ 52.0 MER / 42.9 CBA-HE**.
Inference only — no training, ~3 GB, ~40 min on a T4.

That number is published in the paper, so reproducing it validates decoding,
normalization, word-level language ID, and both metrics end to end. It is **not**
our baseline — it is the calibration of the measuring instrument. A broken metric
makes every downstream result uninterpretable.

Then decode M6/M7/M8 and the `whisper-small` zero-shot baseline they are measured against.

### Decoding reproduces WhisperX, not a per-clip loop
The paper decodes whole recordings with WhisperX. We use **faster-whisper**, the engine
WhisperX wraps: language detected **once per recording**, 30-second chunks with real
context, beam 5, VAD, temperature fallback, and a compression-ratio threshold that aborts
repetition loops.

This matters enormously. Decoding the 3,136 isolated 6-second clips instead makes Whisper
render English loanwords in Devanagari, so the hypothesis contains **no script boundary and
no switch bigram can match**. Measured on real test audio with one model:

| decoding | %Latin in hyp | MER | CBA-HE |
|---|---|---|---|
| reference | 21.6% | – | – |
| per-clip | **0.0%** | 182.6 | 0.0 |
| recording-level | 12.2% | 103.1 | 1.6 |

`utt_id` is `<speaker>_<recording>_<index>`, so the `test` config already on the Hub is
regrouped into its 30 recordings here — no re-upload.

In [ ]:
!pip install -q -U transformers jiwer faster-whisper ctranslate2
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich

# csasr LAST and FORCED: `pip install git+...` treats an already-installed
# version as satisfied and skips the reinstall, so a re-run in a live kernel
# silently keeps OLD code. --no-deps because the deps are installed above.
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

# This NOTEBOOK's own version. `pip install` updates the csasr PACKAGE but NOT the
# .ipynb -- an old notebook against a new package is a real and confusing failure.
NOTEBOOK_VERSION = "0.10.3"

import csasr
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}. "
    "If the PACKAGE is older: restart the kernel (Run > Restart & clear) -- pip skips "
    "a reinstall when the version already looks satisfied. If the NOTEBOOK is older: "
    "re-download it from the repo -- pip does NOT update .ipynb files."
)
print("csasr", csasr.__version__)

import os, subprocess, sys
os.environ["HF_HOME"] = "/kaggle/temp/hf"

from kaggle_secrets import UserSecretsClient
# Put the token in the ENVIRONMENT, never in argv: a CLI arg lands verbatim in
# every traceback and in `ps` output.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])

REAL = "RohanRamesh/mucs-he-cs"
OUT  = "/kaggle/working"

def run(*args):
    """Run a csasr CLI. On failure, re-raise with the LAST LINES of the child's
    output -- otherwise Jupyter shows only `exit status 1` and buries the cause."""
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])   # inherits HF_TOKEN + streams
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed "
            f"ABOVE this traceback - scroll up in this cell's output."
        )

import torch
N_GPU = torch.cuda.device_count()
print(f"{N_GPU} GPU(s) visible")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Decoding: use **both** T4s and batch the chunks

Sequential decoding pinned one GPU at ~4 GB of 15 GB and left the second idle — large-v2
took 1h50m. Two independent fixes:

* **Batched inference** (`--batch-size 16`). VAD carves the recording into speech chunks and
  they are decoded as a batch instead of one at a time: **8× faster**, measured. This is not a
  shortcut — batched VAD inference is exactly what WhisperX does, so it is *more* faithful to
  the paper than our sequential loop was.
* **Shard recordings across the two GPUs**, one process each: another **2×**.

Together: large-v2 ≈ **1h50m → ~8 min**.

`--lang-detect-segments 8` also lands here. faster-whisper detects the language from a *single*
30-second window by default, and one bad window sends a whole recording into Urdu — where no
Hindi/English switch bigram can match and CBA collapses. Voting over 8 windows fixes it.

In [ ]:
from pathlib import Path
from csasr.manifest import read_jsonl, write_jsonl
from csasr.eval.ct2 import resolve_ct2

CT2 = "/kaggle/temp/ct2"

def decode(model, out_name, batch_size=16):
    """Shard the 30 recordings across every GPU, decode in parallel, merge."""
    # Convert ONCE up front: two processes racing on the same cache dir would
    # corrupt it. Prebuilt OpenAI models pass straight through.
    resolve_ct2(model, cache_dir=CT2, quantization="float16")

    n = max(1, N_GPU)
    procs = []
    for i in range(n):
        cmd = [sys.executable, "-m", "csasr.eval.decode",
               "--model", model, "--engine", "faster-whisper", "--mode", "recording",
               "--language", "none", "--batch-size", str(batch_size),
               "--lang-detect-segments", "8",
               "--test-hf", REAL, "--test-config", "test", "--ct2-cache", CT2,
               "--shard", str(i), "--num-shards", str(n),
               "--out", f"{OUT}/{out_name}.shard{i}.jsonl",
               "--refs-out", f"{OUT}/refs.shard{i}.jsonl"]   # PER-UTTERANCE refs
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        print(f"> GPU{i}: {model} shard {i}/{n}", flush=True)
        procs.append(subprocess.Popen(cmd, env=env))

    for i, p in enumerate(procs):
        if p.wait() != 0:
            raise RuntimeError(f"decode shard {i} failed (exit {p.returncode})")

    hyps = [r for i in range(n) for r in read_jsonl(f"{OUT}/{out_name}.shard{i}.jsonl")]
    refs = [r for i in range(n) for r in read_jsonl(f"{OUT}/refs.shard{i}.jsonl")]
    write_jsonl(f"{OUT}/{out_name}.jsonl", hyps)
    write_jsonl(f"{OUT}/refs_utt.jsonl", sorted(refs, key=lambda r: r["utt_id"]))
    print(f"merged {len(hyps)} recordings -> {OUT}/{out_name}.jsonl")
    return hyps

## GATE 3 — calibrate the metric against a published number

Inference only, ~8 min on 2× T4.

In [ ]:
decode("openai/whisper-large-v2", "hyp_largev2_zeroshot")

In [ ]:
from csasr.eval.score import score

REFS = f"{OUT}/refs_utt.jsonl"          # PER-UTTERANCE: CBA's denominator is Table 1's
HYP  = f"{OUT}/hyp_largev2_zeroshot.jsonl"

for mm in ("word", "hybrid"):
    r = score(REFS, HYP, group="recording", mer_mode=mm)
    print(f"MER {mm:<7} {r['mer']:>6.1f}")
print()
for cm in ("adjacent", "lenient"):
    r = score(REFS, HYP, group="recording", cba_mode=cm)
    print(f"CBA {cm:<9} HE {r['cba_he']:>5.1f}   EH {r['cba_eh']:>5.1f}   "
          f"(HE denominator {r['he_total']:,})")

print("\npaper (large-v2 zero-shot): MER 52.0   CBA-HE 42.9   CBA-EH 36.x   HE denom 4,189")
print()
print("MER: 'hybrid' reproduces the paper (51.9 vs 52.0), so they use the SEAME")
print("     definition - characters on Devanagari, words on Latin.")
print("CBA: the paper never defines 'correctly recognized'. 'lenient' lands on their")
print("     42.9; 'adjacent' is the literal bigram reading and lands at half. Both are")
print("     reported. Every system is scored identically, so the M6->M7->M8 ordering")
print("     that Track 2 actually tests is unaffected. See csasr/eval/cba.py.")

### Diagnostics — read these before trusting the numbers above

In [ ]:
from collections import Counter
from csasr.manifest import read_jsonl
from csasr.eval.cba import cba
from csasr.eval.mer import mer
from csasr.lid import Lang, count_words
from csasr.normalize import normalize

from csasr.eval.grouping import concat_refs
refs = concat_refs(list(read_jsonl(f"{OUT}/refs_utt.jsonl")))
hyps = list(read_jsonl(f"{OUT}/hyp_largev2_zeroshot.jsonl"))
R = [refs[h["utt_id"]] for h in hyps]
H = [h["hyp"] for h in hyps]

def mix(t):
    c = count_words(normalize(t, "scoring")); tot = sum(c.values()) or 1
    return c[Lang.HI] / tot, c[Lang.EN] / tot

# 1) BOTH scripts must be present, or CBA is structurally zero regardless of MER.
rh, re_ = mix(" ".join(R)); hh, he = mix(" ".join(H))
print(f"REFERENCE : {rh:5.1%} Devanagari  {re_:5.1%} Latin")
print(f"HYPOTHESIS: {hh:5.1%} Devanagari  {he:5.1%} Latin")

# 2) Hindi/Urdu confusion destroys switch points wholesale.
print("\ndetected language per recording:", dict(Counter(h["detected_language"] for h in hyps)))
def wc(t): return sum(count_words(normalize(t, "scoring")).values())
bad = [h for h in hyps if h["detected_language"] != "hi"]
if bad:
    share = sum(wc(refs[h["utt_id"]]) for h in bad) / sum(wc(t) for t in R)
    print(f"  non-hi: {len(bad)}/{len(hyps)} recordings = {share:.1%} of reference words")
    hi = [h for h in hyps if h["detected_language"] == "hi"]
    ch = cba([refs[h["utt_id"]] for h in hi], [h["hyp"] for h in hi])
    ca = cba(R, H)
    print(f"  CBA-HE  all {ca.he:.1f}  ->  hi-only {ch.he:.1f}")
    print(f"  CBA-EH  all {ca.eh:.1f}  ->  hi-only {ch.eh:.1f}")

# 3) Is any residual MER gap definitional rather than a quality gap?
print()
for p in ("raw", "punct", "scoring"):
    print(f"MER preset={p:8}: {mer(R, H, preset=p):.1f}")

## whisper-small zero-shot — the actual baseline for M6/M7/M8

Expect its CBA ≈ 0: whisper-small transliterates English into Devanagari, so it has no script boundary to match. That is the model, not the pipeline — and it is exactly what fine-tuning is supposed to fix.

In [ ]:
decode("openai/whisper-small", "hyp_small_zeroshot")

## Decode the fine-tuned models

`decode.py` converts each HF checkpoint to CTranslate2 on first use and caches it, so
faster-whisper can load it. Run this **only after** `02_train.ipynb` has pushed the repos —
otherwise you get a 404, which is expected, not a bug.

In [ ]:
for m in ("m6", "m7", "m8"):
    decode(f"RohanRamesh/whisper-small-cs-{m}", f"hyp_{m}")

## Results — reproduce the ordering of Table 2

In [ ]:
systems = [
    ("large-v2 zero-shot", "hyp_largev2_zeroshot.jsonl", 52.0),
    ("small zero-shot",    "hyp_small_zeroshot.jsonl",   None),
    ("M6 (T1, 8h)",        "hyp_m6.jsonl",               48.2),
    ("M7 (T2, 22h)",       "hyp_m7.jsonl",               40.8),
    ("M8 (T2 + mono)",     "hyp_m8.jsonl",               39.2),
]
rows = []
for name, f, paper_mer in systems:
    r = score(f"{OUT}/refs_utt.jsonl", f"{OUT}/{f}", group="recording", mer_mode="hybrid")
    rows.append((name, r, paper_mer))

print(f"{'system':<20}{'MER':>8}{'paper':>8}{'CBA-HE':>9}{'CBA-EH':>9}")
for name, r, pm in rows:
    p = f"{pm:.1f}" if pm else "-"
    print(f"{name:<20}{r['mer']:>8.1f}{p:>8}{r['cba_he']:>9.1f}{r['cba_eh']:>9.1f}")

mers = [r["mer"] for _, r, _ in rows[2:]]
print("\nM6 > M7 > M8 ordering reproduced:", mers == sorted(mers, reverse=True))

with open(f"{OUT}/table2.json", "w") as fh:
    json.dump([{"system": n, **r} for n, r, _ in rows], fh, indent=2)